<a href="https://colab.research.google.com/github/shaheryar2-code/web-rag-pipeline/blob/main/rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain_community langchainhub chromadb langchain langchain-openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

In [50]:
# from google.colab import userdata
# import os

# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAIAPIRKEY')


!pip install -U langchain-google-genai -q

In [53]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('DEMINIAPIKEY')

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [54]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_path=["https://lilianweng.github.io/posts/2023-06-23-agent/"])
docs = loader.load()
print(docs)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final resu

In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)

In [56]:
print (splits[0])
print (splits[1])
print (splits[2])

page_content='LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References' metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The poten

In [57]:
print(len(splits))

66


In [58]:
!pip install -U langchain-chroma chromadb -q

In [59]:
!pip install -U langchain-huggingface sentence-transformers -q

In [61]:

# from langchain_openai import OpenAIEmbeddings
# from langchain_chroma import Chroma

# vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print(vectorstore._collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

132


In [8]:

print(vectorstore._collection.count())

66


In [62]:

print(vectorstore._collection.get())

{'ids': ['ccceecff-2454-40b8-994a-414fd29bc4bc', 'd52619da-a91d-47b7-a67d-18e2e5fd5408', 'fe61abb6-f050-4f56-8a2f-522e2030a3f3', 'fa84f9d1-2672-43f2-b6e5-b3feb071a2b1', 'b41bdf48-6b72-4877-9ace-bf15e51844c2', 'b75613f4-6176-47ea-990a-738c8a6cdff9', '35cc9f20-64c6-4fda-824e-e7e1296227c7', 'e4c92380-5bc0-4a02-98c3-84f6b2c8c7f5', '6a7aed98-fb88-456a-ae9a-ef5ce96b2a63', 'cbeeb039-8a76-439b-9deb-85d07375b7d0', 'b974b754-f6a3-425d-93ff-043e10430e6c', '992e678f-c2da-4ce6-bf62-7bfffa65fbf6', '2baccf66-f1f0-413d-976d-da63a731c147', '26232c24-6cd7-4d48-9345-edade953c409', '6420cf9a-2219-4fce-a621-99eff5bbcd5c', '3b0f0408-b9c9-43b1-94fb-2c93bbf5ebce', '1b08eaff-e5b6-4c78-a4e5-578c9fb83d83', 'c8e000a1-ceda-43bb-b86c-438f42435220', 'cb6bd496-d2f0-4386-9608-98af76c63b93', '8e71b2f6-c93e-46d6-8b9b-5ded76032fb5', 'cbea741d-347f-414d-b8da-6d475f130037', 'be3c1578-470d-45c8-81aa-84a287c3bac7', 'bc05852f-4718-407a-ae7f-98d0c11542cd', '12101209-dba8-413c-b159-288811d95b31', '658ab153-4d2b-4b5b-b3e6-1fb3d4

In [63]:
print("\n collection-1 ", vectorstore._collection.get(ids=['43b7488d-de46-4112-b885-569293a42a81'],include=["embeddings","documents"]))


 collection-1  {'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['embeddings', 'documents'], 'data': None, 'metadatas': None}


In [11]:
print("\n collection-2 ", vectorstore._collection.get(ids=['bf009329-36bf-4b4e-9eaa-7339bd277217'],include=["embeddings","documents"]))

print("\n collection-3 ", vectorstore._collection.get(ids=['13bf36a7-2119-4a09-913a-ee6b65a3bea0 '],include=["embeddings","documents"]))




 collection-2  {'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['embeddings', 'documents'], 'data': None, 'metadatas': None}

 collection-3  {'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['embeddings', 'documents'], 'data': None, 'metadatas': None}


In [64]:
retriever = vectorstore.as_retriever()

In [65]:
!pip install -U langchainhub -q

In [66]:
!pip install -U langsmith -q


In [68]:
from langchain_core.prompts import ChatPromptTemplate

def get_rag_prompt():
    return ChatPromptTemplate.from_template(
        """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say you don't know — don't make one up.
Use three sentences maximum and keep the answer concise.

Context: {context}

Question: {question}

Answer:"""
    )

prompt = get_rag_prompt()

In [69]:
# from langchain_openai import ChatOpenAI

from langchain_google_genai import ChatGoogleGenerativeAI


In [70]:
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

In [71]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [72]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [73]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [74]:
rag_chain.invoke("What is task decomposition?")

'Task decomposition is the process of breaking down a complicated or hard task into smaller, simpler, and more manageable steps so an agent can plan ahead. It can be accomplished through simple LLM prompting, task-specific instructions, human inputs, or external classical planners (LLM+P). Additionally, prompting techniques like Chain of Thought and Tree of Thoughts implement task decomposition by instructing models to think step-by-step or explore multiple reasoning possibilities.'

In [75]:
rag_chain.invoke("What are the approaches to task decomposition?")

'Task decomposition can be done using simple LLM prompting, task-specific instructions, or human inputs. Another approach is LLM+P, which outsources long-horizon planning to an external classical planner using Planning Domain Definition Language (PDDL). Additionally, prompting techniques like Chain of Thought (CoT) decompose tasks step-by-step, while Tree of Thoughts extends CoT by exploring multiple reasoning paths at each step.'

In [76]:
rag_chain.invoke(    "What is the difference between short-term and long-term memory in autonomous agents?")

'In autonomous agents, short-term memory utilizes in-context learning to allow the model to learn within the immediate prompt. In contrast, long-term memory enables the agent to retain and recall information over extended periods. Long-term memory often achieves this capability by leveraging an external vector store and fast retrieval.'

In [77]:
rag_chain.invoke("What are the challenges of building autonomous agents?")

"I don't know. The provided context introduces a section on challenges and limitations of building LLM-powered autonomous agents, but it does not list or describe what those specific challenges are."

In [87]:


from langchain_core.runnables import RunnableLambda

In [88]:


 def print_prompt(prompt_text):
    print("print-", prompt_text)
    return prompt_text

In [91]:
rag_chain_with_print = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | RunnableLambda(print_prompt)
    | llm
    | StrOutputParser()
)

answer = rag_chain_with_print.invoke("What is task decomposition?")
print(answer)

print- messages=[HumanMessage(content='Use the following pieces of context to answer the question at the end.\nIf you don\'t know the answer, just say you don\'t know — don\'t make one up.\nUse three sentences maximum and keep the answer concise.\n\nContext: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.\nAnother quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back int

In [92]:
git init


SyntaxError: invalid syntax (3277417328.py, line 1)